## **A6: Naive RAG vs Contextual Retrieval**

This notebook implements a simplified Retrieval Augmented Generation (RAG) pipeline using a chapter from a Natural Language Processing textbook.

The goal of this assignment is to:
- Extract text from a PDF chapter
- Clean and preprocess the text
- Split the text into manageable chunks
- Create question–answer pairs for evaluation
- Generate embeddings for the text chunks
- Retrieve the most relevant chunks for a given question
- Evaluate the retrieval quality

This pipeline simulates how modern AI systems answer questions by retrieving relevant information from a knowledge source.

In [ ]:
!pip install pymupdf


## **Task 1 – Data Preparation**

In this task, we prepare the knowledge base that will be used for the RAG system.

The steps include:

1. Uploading the PDF chapter containing the study material
2. Extracting the raw text from the PDF
3. Cleaning and normalizing the text
4. Splitting the text into smaller chunks
5. Creating question–answer pairs for evaluation

These chunks will later be embedded and stored so that the system can retrieve the most relevant information when answering questions.

#### **Uploading the Source Document**

The first step is to upload the PDF document that contains the chapter used for building the knowledge base.

This document will later be processed to extract the text and prepare it for retrieval.

In [ ]:
#importing the chapter from files

from google.colab import files
uploaded = files.upload()

Saving 11.pdf to 11 (2).pdf


In [ ]:
# Extracting Text from the PDF
import fitz

pdf_path = "11.pdf"
doc = fitz.open(pdf_path)

raw_text = ""
for page in doc:
    raw_text += page.get_text() + "\n"

print(raw_text[:3000])

Speech and Language Processing.
Daniel Jurafsky & James H. Martin.
Copyright © 2026.
All
rights reserved.
Draft of January 6, 2026.
CHAPTER
11
Information Retrieval and
Retrieval-Augmented Generation
On two occasions I have been asked,—“Pray, Mr. Babbage, if you put into
the machine wrong ﬁgures, will the right answers come out?” ... I am not able
rightly to apprehend the kind of confusion of ideas that could provoke such a
question.
Babbage (1864)
People need to know things. So pretty much as soon as there were computers
we were asking them questions. By 1961 there was a system to answer questions
about American baseball statistics like “How many games did the Yankees play
in July?” (Green et al., 1961). Even ﬁctional computers in the 1970s like Deep
Thought, invented by Douglas Adams in The Hitchhiker’s Guide to the Galaxy,
answered “the Ultimate Question Of Life, The Universe, and Everything”.1 And
because so much knowledge is encoded in text, systems were answering questions
at hum

In [ ]:
# Text Cleaning and Normalization
import re

clean_text = raw_text
clean_text = clean_text.replace("ﬁ", "fi").replace("ﬂ", "fl")
clean_text = re.sub(r'[ \t]+', ' ', clean_text)
clean_text = re.sub(r'\n{3,}', '\n\n', clean_text)
clean_text = clean_text.strip()

print(clean_text[:1500])

Speech and Language Processing.
Daniel Jurafsky & James H. Martin.
Copyright © 2026.
All
rights reserved.
Draft of January 6, 2026.
CHAPTER
11
Information Retrieval and
Retrieval-Augmented Generation
On two occasions I have been asked,—“Pray, Mr. Babbage, if you put into
the machine wrong figures, will the right answers come out?” ... I am not able
rightly to apprehend the kind of confusion of ideas that could provoke such a
question.
Babbage (1864)
People need to know things. So pretty much as soon as there were computers
we were asking them questions. By 1961 there was a system to answer questions
about American baseball statistics like “How many games did the Yankees play
in July?” (Green et al., 1961). Even fictional computers in the 1970s like Deep
Thought, invented by Douglas Adams in The Hitchhiker’s Guide to the Galaxy,
answered “the Ultimate Question Of Life, The Universe, and Everything”.1 And
because so much knowledge is encoded in text, systems were answering questions
at h

#### **Creating Question–Answer Pairs**

To evaluate the retrieval system, we create a set of question–answer pairs based on the content of the chapter.

Each pair contains:

- A **question** related to the chapter
- A **ground truth answer**

These pairs will later be used to measure how well the retrieval system finds relevant information.

In [ ]:
#Question and answer Pairs
qa_pairs = [
{
"question": "What is information retrieval?",
"ground_truth_answer": "Information retrieval is the process of finding relevant documents or passages from a large collection of data in response to a user query."
},
{
"question": "What is the main goal of an information retrieval system?",
"ground_truth_answer": "The main goal of an information retrieval system is to retrieve documents that are most relevant to a user's query from a large document collection."
},
{
"question": "What is a document collection in information retrieval?",
"ground_truth_answer": "A document collection is the set of documents that an information retrieval system searches through to find relevant information."
},
{
"question": "What is a query in information retrieval?",
"ground_truth_answer": "A query is a user's request for information, typically expressed as a set of keywords or a question submitted to an information retrieval system."
},
{
"question": "What is indexing in information retrieval?",
"ground_truth_answer": "Indexing is the process of organizing and storing information from documents in a structured format so that it can be efficiently searched and retrieved."
},
{
"question": "What is an inverted index?",
"ground_truth_answer": "An inverted index is a data structure that maps words or terms to the list of documents in which they appear, allowing efficient retrieval of documents containing specific terms."
},
{
"question": "What is term frequency?",
"ground_truth_answer": "Term frequency refers to the number of times a particular word or term appears in a document."
},
{
"question": "What is inverse document frequency?",
"ground_truth_answer": "Inverse document frequency measures how important a term is by reducing the weight of common words and increasing the weight of rare terms across documents."
},
{
"question": "What is TF-IDF?",
"ground_truth_answer": "TF-IDF is a weighting scheme that combines term frequency and inverse document frequency to evaluate how important a word is to a document in a collection."
},
{
"question": "What is document ranking in information retrieval?",
"ground_truth_answer": "Document ranking is the process of ordering retrieved documents based on their estimated relevance to the user's query."
},
{
"question": "What is dense retrieval?",
"ground_truth_answer": "Dense retrieval uses neural embeddings to represent queries and documents as vectors and retrieves documents based on similarity between these vectors."
},
{
"question": "What is sparse retrieval?",
"ground_truth_answer": "Sparse retrieval represents documents and queries using sparse vectors based on term frequencies or TF-IDF weights."
},
{
"question": "What is the vector space model in information retrieval?",
"ground_truth_answer": "The vector space model represents documents and queries as vectors in a multi dimensional space and calculates similarity between them using measures such as cosine similarity."
},
{
"question": "What is cosine similarity?",
"ground_truth_answer": "Cosine similarity is a metric used to measure the similarity between two vectors by calculating the cosine of the angle between them."
},
{
"question": "What is retrieval augmented generation?",
"ground_truth_answer": "Retrieval augmented generation is a method that combines document retrieval with language models so that the model can generate answers using external knowledge sources."
},
{
"question": "Why is retrieval augmented generation useful for language models?",
"ground_truth_answer": "Retrieval augmented generation allows language models to access up to date and external information, reducing hallucinations and improving factual accuracy."
},
{
"question": "What is chunking in retrieval augmented generation systems?",
"ground_truth_answer": "Chunking is the process of splitting documents into smaller pieces so that relevant sections can be retrieved more effectively during question answering."
},
{
"question": "What role do embeddings play in dense retrieval?",
"ground_truth_answer": "Embeddings convert documents and queries into numerical vector representations so that similarity between them can be computed."
},
{
"question": "What is the role of a retriever in a RAG system?",
"ground_truth_answer": "The retriever identifies and returns relevant document chunks from a knowledge base that can help answer the user's query."
},
{
"question": "What is the role of a generator in a RAG system?",
"ground_truth_answer": "The generator is a language model that produces the final answer by using the retrieved documents as context."
}
]

In [ ]:
import json

with open("qa_pairs.json", "w") as f:
    json.dump(qa_pairs, f, indent=2)

## **Task 2 – Technique Comparison: Naive RAG vs Contextual Retrieval**

In this task, we implement and compare two different retrieval strategies used in Retrieval Augmented Generation (RAG):

1. **Naive RAG**
2. **Contextual Retrieval**

The goal is to determine whether contextual enrichment of text chunks improves retrieval quality and answer generation.

The process includes:

- Building a baseline **Naive RAG pipeline**
- Implementing **Contextual Retrieval with enriched chunk context**
- Running 20 evaluation questions through both pipelines
- Comparing their performance using **ROUGE metrics**

The results will help determine whether contextual information improves semantic retrieval performance.

In [ ]:
# Installing Required Libraries
!pip install sentence-transformers chromadb rouge-score

In [ ]:
!pip install transformers sentencepiece torch

In [ ]:
import json
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

In [ ]:
# Loading chunks from Task- 1

with open("chapter11_chunks.json", "r") as f:
    chunks = json.load(f)

print("Total chunks:", len(chunks))
print("\nExample chunk:\n")
print(chunks[0][:500])

Total chunks: 26

Example chunk:




In [ ]:
# Loading the Question–Answer Dataset

with open("qa_pairs.json", "r") as f:
    qa_pairs = json.load(f)

print("Total questions:", len(qa_pairs))
print("\nExample question:")
print(qa_pairs[0])

Total questions: 20

Example question:
{'question': 'What is information retrieval?', 'ground_truth_answer': 'Information retrieval is the process of finding relevant documents or passages from a large collection of data in response to a user query.'}


In [ ]:
# Generating Text Embeddings (Retriever model)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
chunk_embeddings = embedding_model.encode(chunks)

print("Embedding shape:", chunk_embeddings.shape)

Embedding shape: (26, 384)


### **Naive RAG Implementation**

Naive RAG represents the baseline retrieval approach.

In this method:

1. The document is divided into fixed-size chunks.
2. Each chunk is converted into an embedding.
3. A vector database stores these embeddings.
4. When a question is asked, the retriever finds the most similar chunks.
5. The retrieved chunks are used to generate an answer.

This method does not include additional contextual information beyond the chunk itself.

In [ ]:
import chromadb

client = chromadb.Client()

try:
    client.delete_collection("naive_rag")
except:
    pass

naive_collection = client.create_collection("naive_rag")

chunk_embeddings = embedding_model.encode(chunks)

for i, chunk in enumerate(chunks):
    naive_collection.add(
        documents=[chunk],
        embeddings=[chunk_embeddings[i].tolist()],
        ids=[str(i)]
    )

print("Naive RAG database ready")

Naive RAG database ready


In [ ]:
def retrieve_naive(query, k=3):
    query_embedding = embedding_model.encode([query])[0]

    results = naive_collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=k
    )

    return results["documents"][0]

In [ ]:
# Contextual Retrieval Implementation

contextual_chunks = []

for chunk in chunks:
    contextual_chunk = (
        "This passage is from Chapter 11 about information retrieval, "
        "indexing, ranking, dense retrieval, sparse retrieval, or retrieval "
        "augmented generation.\n\n" + chunk
    )
    contextual_chunks.append(contextual_chunk)

print("Total contextual chunks:", len(contextual_chunks))
print(contextual_chunks[0][:400])

Total contextual chunks: 26
This passage is from Chapter 11 about information retrieval, indexing, ranking, dense retrieval, sparse retrieval, or retrieval augmented generation.




In [ ]:
# Contextual Chunk Enrichment
try:
    client.delete_collection("contextual_rag")
except:
    pass

context_collection = client.create_collection("contextual_rag")

context_embeddings = embedding_model.encode(contextual_chunks)

for i, chunk in enumerate(contextual_chunks):
    context_collection.add(
        documents=[chunk],
        embeddings=[context_embeddings[i].tolist()],
        ids=[str(i)]
    )

print("Contextual retrieval database created")

Contextual retrieval database created


In [ ]:
# Retrieving Relevant Chunks
def retrieve_contextual(query, k=3):
    query_embedding = embedding_model.encode([query])[0]

    results = context_collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=k
    )

    return results["documents"][0]

In [ ]:
!pip install transformers sentencepiece torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

generator_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
generator_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

print("Generator model loaded successfully")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Generator model loaded successfully


In [ ]:
def normalize_question(text):
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

def lookup_saved_answer(question):
    normalized = normalize_question(question)

    for item in results:
        if normalize_question(item["question"]) == normalized:
            return item.get("ground_truth_answer")

    return None

#### **Answer Generation**

After retrieving the most relevant chunks, a language model is used to generate the final answer.

The retrieved chunks serve as supporting context for the model, enabling it to produce accurate responses to the user's question.

This step represents the generation component of the Retrieval Augmented Generation pipeline.

In [ ]:
def generate_answer(question, retrieved_chunks):
    context = "\n\n".join(retrieved_chunks)

    prompt = f"""Context:
{context}

Question: {question}

Give a direct answer based only on the context.
Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    output = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
#creating contexual chunks
contextual_chunks = []

for chunk in chunks:

    contextual_chunk = (
        "This section from Chapter 11 about Information Retrieval explains: "
        + chunk
    )

    contextual_chunks.append(contextual_chunk)

print(contextual_chunks[0][:500])

This section from Chapter 11 about Information Retrieval explains: 


In [ ]:
context_embeddings = embedding_model.encode(contextual_chunks)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
#Running all 20 queries
results = []

for qa in qa_pairs:
    question = qa["question"]
    truth = qa["ground_truth_answer"]

    naive_chunks = retrieve_naive(question, k=3)
    naive_answer = generate_answer(question, naive_chunks)

    context_chunks = retrieve_contextual(question, k=3)
    context_answer = generate_answer(question, context_chunks)

    results.append({
        "question": question,
        "ground_truth_answer": truth,
        "naive_rag_answer": naive_answer,
        "contextual_retrieval_answer": context_answer
    })

print("All questions processed")
print(results[0])

All questions processed
{'question': 'What is information retrieval?', 'ground_truth_answer': 'Information retrieval is the process of finding relevant documents or passages from a large collection of data in response to a user query.', 'naive_rag_answer': 'We’ll begin by introducing information retrieval, the task of choosing the most relevant document from a document set given a user’s query expressing their infor- mation need. We’ll see the classic method based on cosines of sparse tf-idf vec- tors, modern neural ‘dense’ retriev', 'contextual_retrieval_answer': 'Information retrieval Information retrieval or IR is the name of the field encompassing the retrieval of information retrieval'}


In [ ]:
#Results

print(type(results))
print(len(results))
print(results[0])

<class 'list'>
20
{'question': 'What is information retrieval?', 'ground_truth_answer': 'Information retrieval is the process of finding relevant documents or passages from a large collection of data in response to a user query.', 'naive_rag_answer': 'We’ll begin by introducing information retrieval, the task of choosing the most relevant document from a document set given a user’s query expressing their infor- mation need. We’ll see the classic method based on cosines of sparse tf-idf vec- tors, modern neural ‘dense’ retriev', 'contextual_retrieval_answer': 'Information retrieval Information retrieval or IR is the name of the field encompassing the retrieval of information retrieval'}


In [ ]:
#Calculating Rouge Score
scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

naive_scores = []
context_scores = []

for r in results:

    naive_score = scorer.score(
        r["ground_truth_answer"],
        r["naive_rag_answer"]
    )

    context_score = scorer.score(
        r["ground_truth_answer"],
        r["contextual_retrieval_answer"]
    )

    naive_scores.append(naive_score["rouge1"].fmeasure)
    context_scores.append(context_score["rouge1"].fmeasure)

print("Average ROUGE-1 Naive RAG:", np.mean(naive_scores))
print("Average ROUGE-1 Contextual Retrieval:", np.mean(context_scores))

Average ROUGE-1 Naive RAG: 0.17294899114695364
Average ROUGE-1 Contextual Retrieval: 0.16745360183825733


#### **Evaluation Using ROUGE Metrics**

The performance of both retrieval methods is evaluated using ROUGE metrics.

ROUGE measures the similarity between the generated answer and the ground truth answer.

Three metrics are used:

ROUGE-1 measures overlap of individual words.  
ROUGE-2 measures overlap of word pairs.  
ROUGE-L measures similarity based on the longest common subsequence.

Higher scores indicate better answer quality.

In [ ]:
from rouge_score import rouge_scorer
import numpy as np

scorer = rouge_scorer.RougeScorer(
    ['rouge1','rouge2','rougeL'],
    use_stemmer=True
)

naive_r1 = []
naive_r2 = []
naive_rl = []

context_r1 = []
context_r2 = []
context_rl = []

for r in results:

    naive_score = scorer.score(
        r["ground_truth_answer"],
        r["naive_rag_answer"]
    )

    context_score = scorer.score(
        r["ground_truth_answer"],
        r["contextual_retrieval_answer"]
    )

    naive_r1.append(naive_score["rouge1"].fmeasure)
    naive_r2.append(naive_score["rouge2"].fmeasure)
    naive_rl.append(naive_score["rougeL"].fmeasure)

    context_r1.append(context_score["rouge1"].fmeasure)
    context_r2.append(context_score["rouge2"].fmeasure)
    context_rl.append(context_score["rougeL"].fmeasure)

print("Evaluation Table")
print("-------------------------")

print("Naive RAG")
print("ROUGE-1:", np.mean(naive_r1))
print("ROUGE-2:", np.mean(naive_r2))
print("ROUGE-L:", np.mean(naive_rl))

print("\nContextual Retrieval")
print("ROUGE-1:", np.mean(context_r1))
print("ROUGE-2:", np.mean(context_r2))
print("ROUGE-L:", np.mean(context_rl))

Evaluation Table
-------------------------
Naive RAG
ROUGE-1: 0.17294899114695364
ROUGE-2: 0.02630313245336145
ROUGE-L: 0.13276316364681726

Contextual Retrieval
ROUGE-1: 0.16745360183825733
ROUGE-2: 0.025235784991882552
ROUGE-L: 0.13375108485170195


### **Analysis**

The comparison between Naive RAG and Contextual Retrieval demonstrates the impact of contextual chunk enrichment on retrieval performance.

By adding contextual descriptions to each chunk, the retriever gains additional semantic information that may improve the relevance of retrieved documents.

The ROUGE evaluation provides quantitative evidence showing whether contextual retrieval improves answer quality compared to the baseline approach.

| Method | ROUGE-1 | ROUGE-2 | ROUGE-L |
|------|------|------|------|
| Naive RAG | 0.17 | 0.026 | 0.13 |
| Contextual Retrieval | 0.16 | 0.02 | 0.13 |

#### **Discussion of Results**

The evaluation results show that the performance of Naive RAG and Contextual Retrieval is very similar across all ROUGE metrics.

Naive RAG slightly outperformed Contextual Retrieval in **ROUGE-1**  and **ROUGE-2** scores, achieving **0.17** and **0.026** respectively, compared to **0.16** and **0.02** for Contextual Retrieval. This suggests that the baseline retrieval approach was slightly better at matching individual words and short phrases from the ground truth answers.

However, both methods achieved the same **ROUGE-L** score of **0.13**. This indicates that the overall structural similarity between the generated answers and the ground truth answers was nearly identical for both approaches.

One possible reason contextual retrieval did not outperform Naive RAG in this experiment could be the limited size of the dataset and the relatively small number of text chunks used for retrieval. Contextual enrichment is generally more beneficial in larger document collections where additional context helps the retriever better understand the semantic meaning of chunks.

Another factor may be that the original chunks already contained sufficient contextual information, meaning that adding extra context did not significantly improve retrieval quality.

Overall, the results suggest that while contextual retrieval is theoretically designed to improve retrieval performance, in this specific implementation the improvement was minimal. Further experimentation with different chunk sizes, retrieval parameters, or larger document collections may reveal clearer advantages of contextual retrieval.

### **Saving the Evaluation File Content**

The evaluation results are saved as a JSON file.

This file contains all question–answer pairs along with the outputs generated by both retrieval methods.

The JSON file will be submitted as part of the assignment deliverables and allows the instructor to review the generated answers and compare the performance of the two RAG techniques.

In [ ]:
import os
import json

os.makedirs("answer", exist_ok=True)

with open("answer/response-st126671-chapter-11.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("Submission JSON file saved in answer folder")

Submission JSON file saved in answer folder


In [ ]:
with open("answer/response-st126671-chapter-11.json", "r", encoding="utf-8") as f:
    print(f.read()[:1000])

[
  {
    "question": "What is information retrieval?",
    "ground_truth_answer": "Information retrieval is the process of finding relevant documents or passages from a large collection of data in response to a user query.",
    "naive_rag_answer": "We’ll begin by introducing information retrieval, the task of choosing the most relevant document from a document set given a user’s query expressing their infor- mation need. We’ll see the classic method based on cosines of sparse tf-idf vec- tors, modern neural ‘dense’ retriev",
    "contextual_retrieval_answer": "Information retrieval Information retrieval or IR is the name of the field encompassing the retrieval of information retrieval"
  },
  {
    "question": "What is the main goal of an information retrieval system?",
    "ground_truth_answer": "The main goal of an information retrieval system is to retrieve documents that are most relevant to a user's query from a large document collection.",
    "naive_rag_answer": "retrieval-aug

### **Downloading the Results File**

The generated JSON file is downloaded to the local machine.

This file contains the complete evaluation results and will be included in the repository submission as required by the assignment.

In [ ]:
#downloading the answers in answer folder
from google.colab import files
files.download("answer/response-st126671-chapter-11.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import fitz
import re
import json

pdf_path = "11.pdf"   # change only if your PDF file has a different name
doc = fitz.open(pdf_path)

raw_text = ""
for page in doc:
    raw_text += page.get_text("text") + "\n\n"

# clean text but keep paragraph breaks
text = raw_text
text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
text = re.sub(r'[ \t]+', ' ', text)
text = re.sub(r'\n{3,}', '\n\n', text)
text = text.strip()

# split into paragraphs
paragraphs = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 100]

# group paragraphs into chunks
chunks = []
current_chunk = ""

for para in paragraphs:
    if len(current_chunk) + len(para) + 1 <= 1000:
        current_chunk += (" " + para) if current_chunk else para
    else:
        chunks.append(current_chunk.strip())
        current_chunk = para

if current_chunk:
    chunks.append(current_chunk.strip())

print("Total chunks:", len(chunks))
print("\nFirst 3 chunks:\n")
for i in range(min(3, len(chunks))):
    print(f"Chunk {i+1}:")
    print(chunks[i][:500])
    print("-" * 60)

with open("chapter11_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print("chapter11_chunks.json saved")

Total chunks: 26

First 3 chunks:

Chunk 1:

------------------------------------------------------------
Chunk 2:
Speech and Language Processing.
Daniel Jurafsky & James H. Martin.
Copyright © 2026.
All
rights reserved.
Draft of January 6, 2026.
CHAPTER
11
Information Retrieval and
Retrieval-Augmented Generation
On two occasions I have been asked,—“Pray, Mr. Babbage, if you put into
the machine wrong figures, will the right answers come out?” ... I am not able
rightly to apprehend the kind of confusion of ideas that could provoke such a
question.
Babbage (1864)
People need to know things. So pretty much as 
------------------------------------------------------------
Chunk 3:
2
CHAPTER 11
•
RETRIEVAL-BASED MODELS
Simply prompting an LLM is useful for many generation tasks, including those
involving facts. But the fact that knowledge is stored in the feedforward weights of
the LLM leads to a number of problems with prompting as a method for correctly
generating factual texts or answers

In [ ]:
from google.colab import files
files.download("chapter11_chunks.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### **Conclusion**

This assignment implemented and compared two **Retrieval-Augmented Generation (RAG)** approaches: **Naive RAG and Contextual Retrieval**. Both methods were applied to a domain-specific question answering system built from the assigned textbook chapter. The system first processed the chapter text, divided it into chunks, and created embeddings that allowed semantic retrieval of relevant information.

The Naive RAG pipeline retrieved chunks directly from the vector database based on embedding similarity. The Contextual Retrieval pipeline improved this process by enriching each chunk with additional contextual information before generating embeddings.

Based on the ROUGE evaluation results, both approaches produced very similar performance. Naive RAG slightly outperformed Contextual Retrieval in **ROUGE-1 and ROUGE-2** scores, while both methods achieved the same **ROUGE-L** score. This suggests that contextual enrichment did not significantly improve retrieval quality in this experiment.

One possible explanation is that the original chunks already contained enough context for the retriever to locate relevant information. Additionally, the relatively small dataset and limited number of chunks may have reduced the potential benefit of contextual enrichment.

Overall, this experiment demonstrates how different retrieval strategies affect the performance of a RAG system. While Contextual Retrieval is designed to improve retrieval quality, its advantages may become more apparent when applied to larger document collections or more complex knowledge bases.